<a href="https://colab.research.google.com/github/sitthinon-stat/Sports-court-rental/blob/rental-1/Sports_court_rental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sports-court-rental
ไว้สำหรับการจองสนามกีฬา จาก Project อาจารย์พิชญา


## ส่วนที่ 1 — เตรียม class และฟังก์ชัน

In [62]:
import random
import time
random.seed(7)
import datetime
import pandas as pd

In [63]:
class Customer:
#class 1 ลูกค้าจองสนามกีฬา
  def __init__(self, customer_id, name, phone_number, is_member=False):
    self.customer_id = customer_id
    self.name = name
    self.phone_number = phone_number
    self.is_member = is_member

  def register_member(self):
    self.is_member = True
    print(f'{self.name} สมัครสมาชิกสำเร็จ><')

class Court:
#class 2 สนามกีฬา
  def __init__(self, court_id, sport_type, hourly_rate, court_status=True):
    self.court_id = court_id
    self.sport_type = sport_type
    self.hourly_rate = hourly_rate
    self.court_status = court_status

  def set_court_unavailable(self):
    self.court_status = False
    print(f'สนาม {self.court_id} ถูกจองหรือปิดปรับปรุง')

class Equipment:
#class 3 อุปกรณ์เสริมให้เช่า
  def __init__(self, equipment_id, equipment_name, equipment_price, stock_quantity):
    self.equipment_id = equipment_id
    self.equipment_name = equipment_name
    self.equipment_price = equipment_price  # Corrected typo: prive -> price
    self.stock_quantity = stock_quantity

  def reduce_stock(self, amount):
    if self.stock_quantity >= amount:
      self.stock_quantity-= amount
      return True
    print(f"อุปกรณ์ {self.equipment_name} หมด")
    return False

  def return_stock(self, amount):
    self.stock_quantity += amount

class Booking:
#class 4 การจองสนาม
  def __init__(self, booking_id, customer, court, start_time, hours):
    self.booking_id = booking_id
    self.customer = customer
    self.court = court
    self.start_time = start_time
    self.hours = hours
    self.equipment_list = []
    self.status = "Confirm"

  def add_equipment(self, equipment, quantity):
    if equipment.reduce_stock(quantity):
        self.equipment_list.append((equipment, quantity))
        print(f"เพิ่ม {equipment.equipment_name} จำนวน {quantity} ชิ้น ในการจองแล้ว")

  def calculate_total_price(self):
        court_price = self.court.hourly_rate * self.hours
        equipment_price = sum(
            item.equipment_price * qty for item, qty in self.equipment_list # Corrected: item.rental_price -> item.equipment_price
        )

        total = court_price + equipment_price
        return total

class Payment:
#class 5 การชำระเงิน
  def __init__(self, payment_id, booking,  payment_method):
    self.payment_id = payment_id
    self.booking = booking
    self.payment_method = payment_method
    # The amount is calculated here, so it relies on Booking.calculate_total_price
    self.amount = self.booking.calculate_total_price()
    self.payment_status = "Pending"

  def process_payment(self):
        self.payment_status = "Paid"
        print(
            f"ชำระเงินสำเร็จ: รหัส {self.payment_id} ยอดเงิน {self.amount} บาท ผ่าน {self.payment_method}"
        )

In [84]:
def generate_thai_name():
    # 1.ฟังก์ชัน: สุ่มชื่อ-นามสกุลลูกค้า -> คืนค่าเป็น string
    first_names = [
        "สมชาย", "สมหญิง", "วิชัย", "อรุณี", "ปรีชา",
        "มานี", "กิตติ", "ศิริพร", "ธนากร", "นภัสสร",
        "ธีรภัทร", "ณัฐวุฒิ", "สุพรรษา", "วรวิทย์", "ชลธิชา","ชินานาง","สิทธินนท์"
    ]

    last_names = [
        "ใจดี", "รักเรียน", "สายทอง", "ศรีสุข", "มั่นคง",
        "เจริญสุข", "วงศ์สว่าง", "รัตนไพศาล", "สุขเกษม", "พัฒนากุล",
        "แสงสุริยา", "ทองประเสริฐ", "เกียรติขจร", "ประสิทธิ์โชค", "บุญรักษา","เชื้อกุณะ","โยธาธรณ์"
    ]

    return f"{random.choice(first_names)} {random.choice(last_names)}"


def generate_phone_number(prefix_list=None):
  # 2.ฟังก์ชันเสริม: สุ่มเบอร์โทรศัพท์มือถือไทย 10 หลัก เพื่อใช้สร้าง Customer
    prefixes = prefix_list if prefix_list is not None else ["081", "086", "089", "092", "095", "061"]
    suffix = "".join([str(random.randint(0, 9)) for _ in range(7)])
    prefix = random.choice(prefixes)
    return f"{prefix}-{suffix[:3]}-{suffix[3:]}"


def random_booking_hours(min_hours=1, max_hours=4):
    # 3.ฟังก์ชัน: สุ่มจำนวนชั่วโมงที่ต้องการจองสนาม (1 - 4 ชั่วโมง) -> คืนค่าเป็น int
    return random.randint(min_hours, max_hours)


def random_equipment_qty(min_qty=1, max_qty=4):
    # 4.ฟังก์ชัน: สุ่มจำนวนอุปกรณ์เสริมที่ต้องการเช่า (คืนค่าเป็น int)
    return random.randint(min_qty, max_qty)


def random_start_time(min_hour=8, max_hour=21):
  # 5.ฟังก์ชันเสริม: สุ่มเวลาเริ่มเล่น (ช่วง 08:00 - 21:00 น.)
    hour = random.randint(min_hour, max_hour)
    minute = random.choice(["00", "30"])
    return f"{hour:02d}:{minute}"


def format_currency(amount, symbol="บาท"):
    # 6.ฟังก์ชัน: จัดรูปแบบตัวเลขเป็นสตริงราคาพร้อมเครื่องหมายจุลภาค -> คืนค่าเป็น string
    return f"{amount:,.2f} {symbol}"

## ส่วนที่ 2 — ทดสอบฟังก์ชันทีละตัว ก่อนเอาไปประกอบเป็นกระบวนการ

ก่อนจะเอาฟังก์ชันไปใช้จริง ควรลองเรียกดูเฉย ๆ ทีละตัวก่อน เพื่อดูว่า **input ที่ใส่เข้าไป**
กับ **output ที่ได้กลับมา** ตรงกับที่ออกแบบไว้หรือไม่


In [82]:
# 1.เรียก generate_thai_name() 8 ครั้ง -> ทุกครั้งได้ชื่อสุ่มไม่ซ้ำแบบ (แสดงว่าฟังก์ชันทำงานทุกครั้งที่เรียก)
for _ in range(10):
    print("ชื่อที่สุ่มได้:", generate_thai_name())

ชื่อที่สุ่มได้: ธีรภัทร แสงสุริยา
ชื่อที่สุ่มได้: ณัฐวุฒิ เชื้อกุณะ
ชื่อที่สุ่มได้: ชลธิชา สายทอง
ชื่อที่สุ่มได้: วิชัย สุขเกษม
ชื่อที่สุ่มได้: ชินานาง สายทอง
ชื่อที่สุ่มได้: สมหญิง พัฒนากุล
ชื่อที่สุ่มได้: ชลธิชา พัฒนากุล
ชื่อที่สุ่มได้: สุพรรษา ทองประเสริฐ
ชื่อที่สุ่มได้: สมชาย บุญรักษา
ชื่อที่สุ่มได้: ณัฐวุฒิ เจริญสุข


In [83]:
# 2.ตัวอย่างการเรียกใช้งานและการเปรียบเทียบ การสุ่มเบอร์โทรศัพท์
print("เรียกแบบ default (ไม่ใส่ argument):", f"{generate_phone_number()}")
print("เรียกแบบระบุกลุ่มเบอร์เอง (เฉพาะ 089, 099):", f"{generate_phone_number(['089', '099'])}")
print("เรียกแบบระบุแค่ prefix_list (keyword argument):", f"{generate_phone_number(prefix_list=['061', '062'])}")

เรียกแบบ default (ไม่ใส่ argument): 061-917-0342
เรียกแบบระบุกลุ่มเบอร์เอง (เฉพาะ 089, 099): 099-366-7127
เรียกแบบระบุแค่ prefix_list (keyword argument): 062-842-6846


In [66]:
# 3.ตัวอย่างการเรียกใช้งานและการเปรียบเทียบ จำนวนชม.การจอง
print("เรียกแบบ default (ไม่ใส่ argument):", f"{random_booking_hours()} ชั่วโมง")
print("เรียกแบบระบุช่วงเอง (2 - 5 ชม.):", f"{random_booking_hours(2, 5)} ชั่วโมง")
print("เรียกแบบระบุแค่ max_hours (keyword argument):", f"{random_booking_hours(max_hours=3)} ชั่วโมง")

เรียกแบบ default (ไม่ใส่ argument): 2 ชั่วโมง
เรียกแบบระบุช่วงเอง (2 - 5 ชม.): 2 ชั่วโมง
เรียกแบบระบุแค่ max_hours (keyword argument): 3 ชั่วโมง


In [71]:
# 4.ตัวอย่างการเรียกใช้งานและการเปรียบเทียบ จำนวนอุปกรณ์เสริมที่ต้องการเช่า
print("เรียกแบบ default (ไม่ใส่ argument):", f"{random_equipment_qty()} ชิ้น")
print("เรียกแบบระบุช่วงเอง (1 - 2 ชิ้น):", f"{random_equipment_qty(1, 2)} ชิ้น")
print("เรียกแบบระบุแค่ max_qty (keyword argument):", f"{random_equipment_qty(max_qty=2)} ชิ้น")

เรียกแบบ default (ไม่ใส่ argument): 4 ชิ้น
เรียกแบบระบุช่วงเอง (1 - 2 ชิ้น): 1 ชิ้น
เรียกแบบระบุแค่ max_qty (keyword argument): 1 ชิ้น


In [97]:
# 5.ตัวอย่างการเรียกใช้งานและการเปรียบเทียบ ช่วงเวลาเริ่มเล่น
print("เรียกแบบ default (ไม่ใส่ argument):", f"{random_start_time()} น.")
print("เรียกแบบระบุช่วงเอง (17:00 - 22:00 น.):", f"{random_start_time(17, 22)} น.")
print("เรียกแบบระบุแค่ max_hour (keyword argument):", f"{random_start_time(max_hour=12)} น.")

เรียกแบบ default (ไม่ใส่ argument): 21:00 น.
เรียกแบบระบุช่วงเอง (17:00 - 22:00 น.): 21:00 น.
เรียกแบบระบุแค่ max_hour (keyword argument): 09:30 น.


In [67]:
# 6.จำลองยอดชำระจากการคำนวณราคาจองสนามและอุปกรณ์ (raw_price จาก calculate_total_price)
raw_booking_price = 58990.0

print("ก่อนจัดรูปแบบ:", raw_booking_price, type(raw_booking_price))
print("หลังจัดรูปแบบ (ค่าเริ่มต้น):", format_currency(raw_booking_price), type(format_currency(raw_booking_price)))
print("หลังจัดรูปแบบ (ระบุหน่วยเอง):", format_currency(raw_booking_price, symbol="THB"))

ก่อนจัดรูปแบบ: 58990.0 <class 'float'>
หลังจัดรูปแบบ (ค่าเริ่มต้น): 58,990.00 บาท <class 'str'>
หลังจัดรูปแบบ (ระบุหน่วยเอง): 58,990.00 THB


## ส่วนที่ 3 — ฟังก์ชันอธิบายขั้นตอนคำนวณราคา (ให้เห็นว่าฟังก์ชันเรียกฟังก์ชัน/method อื่นต่อได้)

In [61]:
## ส่วนที่ 3 — ฟังก์ชันอธิบายขั้นตอนคำนวณราคาจองสนามและอุปกรณ์
def explain_price_calculation(booking):
    """รับ object Booking 1 ตัว -> พิมพ์อธิบายการคำนวณราคาทีละขั้นตอน"""
    print(f"=== รายละเอียดการคำนวณราคา รหัสการจอง: {booking.booking_id} ===")
    print(f"ผู้จอง: {booking.customer.name} ({'สมาชิก' if booking.customer.is_member else 'บุคคลทั่วไป'})")

    # 1. คำนวณค่าสนาม
    court_rate = booking.court.hourly_rate
    hours = booking.hours
    court_total = court_rate * hours
    print(f"1. ค่าสนาม ({booking.court.sport_type}):")
    print(f"   อัตรา {court_rate} บาท/ชม. x {hours} ชม. = {format_currency(court_total)}")

    # 2. คำนวณค่าอุปกรณ์เสริม (ถ้ามี)
    print("2. ค่าอุปกรณ์เสริมที่เช่า:")
    if not booking.equipment_list:
        print("   (ไม่มีการเช่าอุปกรณ์เสริม)")
        equipment_total = 0
    else:
        equipment_total = 0
        for item, qty in booking.equipment_list:
            item_subtotal = item.equipment_price * qty
            equipment_total += item_subtotal
            print(f"   - {item.equipment_name} ({item.equipment_price} บาท x {qty} ชิ้น) = {format_currency(item_subtotal)}")
        print(f"   รวมค่าอุปกรณ์ = {format_currency(equipment_total)}")

    # 3. คำนวณราคาสุทธิโดยเรียก method ของ Booking
    final_price = booking.calculate_total_price()
    print("--------------------------------------------------")
    print(f"ราคารวมสุทธิทั้งหมด = {format_currency(final_price)}")
    print("==================================================")

    return final_price



## ส่วนที่ 4 — จำลอง "ลูกค้า 1 คนเดินเข้าร้าน" แบบ step-by-step



In [98]:
## ส่วนที่ 4 — จำลอง "ลูกค้า 1 คนเดินเข้าร้าน" แบบ step-by-step
def simulate_customer_visit(booking_id, customer, courts_list, equipments_list, pause=0.0):
    """
    จำลองขั้นตอนทั้งหมดตอนลูกค้าเดินเข้ามาจองสนามที่เคาน์เตอร์ แล้วคืนค่า object Booking
    - booking_id: รหัสการจอง
    - customer: object Customer
    - courts_list: รายการ object Court ทั้งหมดที่มี
    - equipments_list: รายการ object Equipment ทั้งหมดที่มี
    - pause: หน่วงเวลาระหว่างขั้นตอน (วินาที) สำหรับ Live Demo
    """
    print("=" * 65)
    print(f"🚪 รายการ #{booking_id}: ลูกค้า '{customer.name}' เดินเข้ามาที่เคาน์เตอร์ "
          f"({'สมาชิก' if customer.is_member else 'ลูกค้าทั่วไป'})")
    print(f"📞 เบอร์โทรศัพท์: {customer.phone_number}")
    time.sleep(pause)

    # 1. พนักงานสอบถามประเภทกีฬา และเลือกสนาม
    chosen_court = random.choice(courts_list)
    hours = random_booking_hours(1, 3)
    start_time = random_start_time()

    print(f"🏟️ ลูกค้าต้องการเล่นกีฬา: {chosen_court.sport_type} (สนามรหัส: {chosen_court.court_id})")
    print(f"⏱️ เวลาที่ต้องการเริ่ม: {start_time} น. | จำนวน: {hours} ชั่วโมง")
    time.sleep(pause)

    # 2. สร้าง Object Booking
    booking = Booking(booking_id, customer, chosen_court, start_time, hours)
    print(f"📋 ระบบสร้างข้อมูลการจองสำเร็จ (สถานะ: '{booking.status}')")
    time.sleep(pause)

    # 3. พนักงานสอบถามเรื่องการเช่าอุปกรณ์เสริม (สุ่มโอกาสเช่า 70%)
    if equipments_list and random.random() < 0.7:
        # สุ่มเลือกอุปกรณ์ 1 ชนิด และสุ่มจำนวน 1-2 ชิ้น
        selected_eq = random.choice(equipments_list)
        qty_to_rent = random_equipment_qty(1, 2)
        print(f"🏸 ลูกค้าต้องการเช่าอุปกรณ์เสริม: {selected_eq.equipment_name} จำนวน {qty_to_rent} ชิ้น")
        booking.add_equipment(selected_eq, qty_to_rent)
    else:
        print("🏸 ลูกค้าไม่ได้เช่าอุปกรณ์เสริมเพิ่มเติม")
    time.sleep(pause)

    # 4. คำนวณราคาและแจกแจงค่าใช้จ่าย
    print("\n💰 การคำนวณราคา:")
    final_price = explain_price_calculation(booking)
    time.sleep(pause)

    # 5. ชำระเงินและออกใบเสร็จ
    payment_methods = ["เงินสด (Cash)", "PromptPay QR", "บัตรเครดิต (Credit Card)"]
    chosen_method = random.choice(payment_methods)
    payment = Payment(f"PAY-{booking_id}", booking, chosen_method)
    payment.process_payment()
    time.sleep(pause)

    # อัปเดตสถานะสนาม
    chosen_court.set_court_unavailable()

    print(f"\n🧾 ใบเสร็จรับเงิน #{payment.payment_id}")
    print(f"   ลูกค้า: {customer.name} | สนาม: {chosen_court.court_id} ({chosen_court.sport_type})")
    print(f"   เวลา: {booking.start_time} น. ({booking.hours} ชม.) | ชำระผ่าน: {chosen_method}")
    print(f"   ยอดชำระสุทธิ: {format_currency(final_price)}")
    print("=" * 65)

    return booking